# Instamart Customer Intelligence & Support Copilot — walkthrough notebook

This notebook runs every part of the project in order and shows the printed results, so you can read them step by step. It uses exactly the same code as the scripts in `src/`.

**Notes**
- This notebook is optional. The graded deliverables are the scripts, the outputs and `README.md`.
- It produces **text only** (no charts), following the brief's "text only" rule.
- To run it in VS Code, select the `.venv` kernel. If VS Code asks for `ipykernel`, install it with `python -m pip install ipykernel` (it is only needed for the notebook, so it is not in `requirements.txt`).

## Setup
The notebook lives in `notebooks/`, but all paths start from the main project folder, so we move one folder up.

In [1]:
import os
import subprocess
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Working in:", os.getcwd())

Working in: c:\swiggy_instamart_copilot\instamart_copilot


## Generate the raw data
Runs the supplied `generate_data.py` (unchanged). The last line must be `generated: customers=1200 products=40 orders_raw=13084 tickets=164`.

In [2]:
result = subprocess.run([sys.executable, "generate_data.py"], capture_output=True, text=True, check=True)
print(result.stdout)

generated: customers=1200 products=40 orders_raw=13084 tickets=164



## Part A — Data foundation
### Tasks 2–5: audit, clean and store

In [3]:
from src import data_pipeline
data_pipeline.main()


=== customers ===
shape: (1200, 5)
            dtype  missing_count  missing_pct
customer_id   str              0         0.00
city          str              0         0.00
signup_date   str              0         0.00
age           str             61         5.08
gender        str             30         2.50

=== products ===
shape: (40, 4)
              dtype  missing_count  missing_pct
product_id      str              0          0.0
product_name    str              0          0.0
category        str              0          0.0
price_inr     int64              0          0.0

=== orders ===
shape: (13084, 9)
                  dtype  missing_count  missing_pct
order_id            str              0         0.00
customer_id         str              0         0.00
product_id          str              0         0.00
quantity          int64              0         0.00
order_ts            str              0         0.00
status              str              0         0.00
discount_pct     

### Task 5: validation

In [4]:
from src import validate
validate.main()

ALL CHECKPOINTS PASSED


### Task 6: the seven SQL queries

In [5]:
from src import sql_analysis
sql_analysis.main()

Q1 customers with completed orders > 500 adding up to > 8000: 30
Q2 rows: 30 | total revenue: INR 2998668.8
   largest cell: Bengaluru / Electronics INR 233909.45
Q3 products above the average price: 11
Q4 customers with no completed order: 25
Q5 customers above the average spend: 550 | average: INR 2552.06
Q6 top customer per city:
     city customer_id  total_spend
Bengaluru       C0082     12856.20
  Chennai       C0930     10151.40
    Delhi       C0291     13742.60
Hyderabad       C0743     12730.05
   Mumbai       C0693     14766.55
     Pune       C0426     10377.10
   rows where RANK() differs from ROW_NUMBER(): 8
Q7 daily rows: 183 | last cumulative value: INR 2998668.8


### Task 7: pandas cross-check and statistics

In [6]:
from src import eda
eda.main()

Pivot table (INR):
category       Dairy  Electronics    Grocery  Personal Care     Snacks         All
city                                                                              
Bengaluru   87075.05    233909.45  110915.65       20193.85   71549.40   523643.40
Chennai     78301.80    195508.80   93817.45       27675.40   68325.70   463629.15
Delhi      117271.70    158549.10  112658.05       23471.50   90879.45   502829.80
Hyderabad   83731.00    228913.10  109475.40       24800.05   81670.35   528589.90
Mumbai      75209.85    214738.40   78970.15       38018.00   72122.10   479058.50
Pune        90349.55    187629.25  108456.60       34275.15   80207.50   500918.05
All        531938.95   1219248.10  614293.30      168433.95  464754.50  2998668.80
pandas grand total: 2998668.8 | SQL total: 2998668.8 | equal: True

Per city:
               total    mean  count
city                               
Bengaluru  523643.40  276.47   1894
Chennai    463629.15  272.72   1700
Delhi      5

**My observations:** *Order amounts are right-skewed (mean 269.91 vs median 135.00), with 897 high outliers that are mostly real electronics orders, so I kept them. All correlations are very weak (the strongest is amount vs discount at −0.095), and correlation does not mean causation.*

## Part B — Customer segments and lapse prediction
This cell takes about a minute because of the two grid searches.

In [ ]:
from src import customer_models
customer_models.main()

**My observations:** *(why K = 4, what each segment means, which model you chose and why)*

## Part C — Support ticket triage

In [ ]:
from src import triage
triage.main()

## Part D — Policy RAG assistant

In [ ]:
from src import policy_rag
policy_rag.main()

Try your own question:

In [ ]:
rag = policy_rag.PolicyRAG()
result = rag.answer("Can wallet balance be refunded as cash?")
print(result["answer"])
print("Sources:", result["sources"])

## Part D — Refund agent

In [ ]:
from src import refund_agent
refund_agent.main()

Try one request yourself:

In [ ]:
result = refund_agent.run_agent("Please refund order ORD-100067", "my-test")
print(result["status"], "|", result["final_answer"])
print("tools:", [step["tool"] for step in result["steps"]])

## Conclusion
*(Write 3–5 sentences: what you built, the most interesting result, and what you would improve.)*